# OPF head-only SFT on Colab T4

This notebook fine-tunes OpenAI Privacy Filter's output head on a curated span dataset.

**Approach: calibration first, then SFT.**

1. **Viterbi calibration** (v5) recovered 8 false negatives for 2 new false positives — free recall gains.
2. **Head-only SFT** targets the remaining literal/multilingual/spelled-out/obfuscated identifier failures that calibration cannot reach.
3. Quasi-identifiers and indirect requests are **excluded** from training — they require semantic reasoning that head-only training cannot learn, and their whole-text span annotations would poison the output head.

Important:
- This dataset is **head-only-safe** — only contains cases where token-level signal exists.
- Because Colab T4 is memory-constrained, this uses a **head-only OPF finetune**, not the stock full-model `opf train` path.
- The hard negative set includes all baseline FP cases plus calibration-induced FP to prevent precision regression.


## Optional first pass: decoder calibration

Before training, try a short decoder-calibration sweep.

- It is much cheaper than SFT.
- It only changes Viterbi transition biases, so it helps precision/recall tradeoffs but does not teach new semantics.
- In this repo, `detect_pii.py` supports `--opf-viterbi-calibration-path` directly.

Practical workflow:

1. Create or edit a local calibration JSON.
2. Run `python detect_pii.py --opf-viterbi-calibration-path <path> ...`.
3. If the benchmark still misses policy cases, continue to SFT.

Example benchmark run:

```bash
python detect_pii.py   --opf-viterbi-calibration-path calib/recall.json   --output results/privacy-filter-calib.json   --verbose
```

Example calibration JSON:

```json
{
  "operating_points": {
    "default": {
      "biases": {
        "transition_bias_background_stay": -0.25,
        "transition_bias_background_to_start": 0.25,
        "transition_bias_inside_to_continue": 0.15,
        "transition_bias_inside_to_end": 0.0,
        "transition_bias_end_to_background": 0.0,
        "transition_bias_end_to_start": 0.0
      }
    }
  }
}
```

Heuristic direction:

- More recall: lower `transition_bias_background_stay`, raise `transition_bias_background_to_start` and `transition_bias_inside_to_continue`.
- More precision: move those settings in the opposite direction.


In [ ]:
!nvidia-smi


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ivanreeve/vllm-qwen3guard-test.git"
BRANCH = "feature/openai-privacy-filter"
WORKDIR = Path("/content/vllm-qwen3guard-test")


In [ ]:
if WORKDIR.exists():
    subprocess.run(["rm", "-rf", str(WORKDIR)], check=True)
subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
print("cwd:", WORKDIR)


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
import torch

print("cuda_available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("This notebook expects a CUDA runtime for training.")

gpu = torch.cuda.get_device_properties(0)
gpu_mem_gb = gpu.total_memory / (1024 ** 3)
print("gpu_name:", gpu.name)
print("gpu_mem_gb:", round(gpu_mem_gb, 1))


## Training recipe

The default recipe below is intentionally T4-safe:

- custom binary label space: `O` / `pii`
- train on literal/multilingual/obfuscated/spelled-out identifier failures + calibration-recovered reinforcement anchors + support anchors
- hard negatives include all baseline FP + calibration-induced FP (TC-136, TC-145)
- validate on a disjoint support set
- disable OPF Triton kernels for the first run (`OPF_MOE_TRITON=0`) to reduce environment brittleness
- use `n_ctx=256` because these examples are short and long context only wastes memory
- freeze the OPF backbone and train only the output head

If the model underfits, raise `--epochs` first.
If it still misses literal-token cases, add more support anchors before raising LR.

**Do NOT add quasi-identifiers or indirect requests** — their whole-text span annotations will teach the head to fire on common words and blow up false positives.


In [ ]:
subprocess.run(["python", "scripts/build_opf_policy_sft_assets.py"], check=True)

for path in [
    "data/opf_policy_sft_train.jsonl",
    "data/opf_policy_sft_val.jsonl",
    "data/opf_policy_label_space.json",
    "data/opf_policy_sft_manifest.json",
]:
    print(path, "exists =", Path(path).exists())


In [ ]:
os.environ["OPF_MOE_TRITON"] = "0"

subprocess.run(
    [
        "python",
        "scripts/train_opf_head_only.py",
        "data/opf_policy_sft_train.jsonl",
        "--validation-dataset",
        "data/opf_policy_sft_val.jsonl",
        "--label-space-json",
        "data/opf_policy_label_space.json",
        "--device",
        "cuda",
        "--n-ctx",
        "256",
        "--epochs",
        "25",
        "--batch-size",
        "1",
        "--grad-accum-steps",
        "8",
        "--learning-rate",
        "2e-4",
        "--weight-decay",
        "0.0",
        "--max-grad-norm",
        "1.0",
        "--output-dir",
        "checkpoints/opf_policy_sft_v1",
        "--overwrite-output",
    ],
    check=True,
)


In [ ]:
# 1. Calibration-only baseline (no SFT) for comparison
subprocess.run(
    [
        "python",
        "detect_pii.py",
        "--opf-viterbi-calibration-path",
        "calib/recall_v5.json",
        "--output",
        "results/privacy-filter-calib-v5.json",
        "--verbose",
    ],
    check=True,
)

# 2. SFT checkpoint (inherits calibration if baked into checkpoint)
subprocess.run(
    [
        "python",
        "detect_pii.py",
        "--model",
        "checkpoints/opf_policy_sft_v1",
        "--output",
        "results/privacy-filter-sft.json",
        "--verbose",
    ],
    check=True,
)


In [ ]:
import json
from pathlib import Path

print("=" * 60)
print("CALIBRATION-ONLY BASELINE (v5)")
print("=" * 60)
calib = json.loads(Path("results/privacy-filter-calib-v5.json").read_text())
print(json.dumps(calib["metrics"], indent=2))
calib_failed = [r["id"] for r in calib["results"] if r["expected"] != r["predicted"]]
print("num_failed:", len(calib_failed))

print()
print("=" * 60)
print("CALIBRATION + HEAD-ONLY SFT")
print("=" * 60)
sft = json.loads(Path("results/privacy-filter-sft.json").read_text())
print(json.dumps(sft["metrics"], indent=2))
sft_failed = [r["id"] for r in sft["results"] if r["expected"] != r["predicted"]]
print("num_failed:", len(sft_failed))

print()
print("=" * 60)
print("DELTA")
print("=" * 60)
for key in ["accuracy", "precision", "recall", "f1"]:
    delta = sft["metrics"][key] - calib["metrics"][key]
    print(f"  {key}: {delta:+.4f}")

calib_fn = {r["id"] for r in calib["results"] if r["expected"] and not r["predicted"]}
sft_fn = {r["id"] for r in sft["results"] if r["expected"] and not r["predicted"]}
recovered = sorted(calib_fn - sft_fn)
regressed = sorted(sft_fn - calib_fn)
print(f"  recovered_fn: {recovered}")
print(f"  regressed_fn: {regressed}")
